In [0]:
%sql
CREATE OR REPLACE TABLE automobilerepair.gold.cube_kpi_metrics AS
SELECT 
    -- Primary Keys
    o.orderlifecycle_id,
    o.order_id,
    
    -- Store Dimensions
    o.store_id,
    s.store_name,
    s.manager_id,
    s.manager_name,
    s.store_type,
    
    -- Technician Dimensions
    o.technician_id,
    t.technician_name,
    
    -- Service Information
    o.service_type,
    o.order_status,
    
    -- Vehicle Information
    o.vehicle_no,
    
    -- Date/Time Fields
    o.vehicle_in_datetime,
    o.vehicle_out_datetime,
    o.actual_work_start_datetime,
    o.actual_completion_datetime,
    o.promised_delivery_datetime,
    o.actual_delivery_datetime,
    o.invoice_date,
    
    -- Financial Fields
    o.invoice_id,
    o.invoice_amount,
    o.estimate_id,
    o.estimate_amount,
    b.budget_amount,
    
    -- Estimator Dimensions
    e.estimator_id,
    e.estimator_name,
    e.estimate_type,
    
    -- Survey Fields
    cs.survey_id,
    cs.responded_flag AS survey_responded_flag,
    cs.delivered_on_time_rating,
    cs.work_quality_rating,
    cs.cleanliness_rating,
    cs.communication_rating,
    
    -- Calculated Metrics: Time Durations (in days)
    DATEDIFF(DAY, o.vehicle_in_datetime, o.vehicle_out_datetime) AS days_in_shop,
    DATEDIFF(DAY, o.vehicle_in_datetime, o.actual_work_start_datetime) AS days_vehicle_in_to_work_start,
    DATEDIFF(DAY, o.actual_work_start_datetime, o.actual_completion_datetime) AS days_work_start_to_completion,
    DATEDIFF(DAY, o.actual_completion_datetime, o.actual_delivery_datetime) AS days_completion_to_delivery,
    
    -- Calculated Metrics: Completion Time Accuracy (in days, negative means early, positive means late)
    DATEDIFF(DAY, o.promised_delivery_datetime, o.actual_delivery_datetime) AS completion_time_variance_days,
    ABS(DATEDIFF(DAY, o.promised_delivery_datetime, o.actual_delivery_datetime)) AS completion_time_accuracy_abs_days,
    
    -- Calculated Metrics: Estimate Accuracy
    CASE 
        WHEN o.estimate_amount > 0 THEN 
            ABS(o.invoice_amount - o.estimate_amount) * 100.0 / o.estimate_amount 
        ELSE NULL 
    END AS estimate_variance_pct,
    
    -- Calculated Metrics: Survey Overall Satisfaction (average of all ratings)
    CASE 
        WHEN cs.responded_flag = true THEN 
            (COALESCE(cs.delivered_on_time_rating, 0) + 
             COALESCE(cs.work_quality_rating, 0) + 
             COALESCE(cs.cleanliness_rating, 0) + 
             COALESCE(cs.communication_rating, 0)) / 4.0
        ELSE NULL 
    END AS overall_satisfaction_rating,
    
    -- Date Dimensions for Time-based Analysis
    DATE_TRUNC('month', o.invoice_date) AS invoice_month,
    YEAR(o.invoice_date) AS invoice_year,
    MONTH(o.invoice_date) AS invoice_month_num,
    DATE_FORMAT(o.invoice_date, 'yyyy-MM') AS invoice_year_month,
    
    -- Flags for Filtering
    CASE WHEN o.order_status = 'COMPLETED' THEN 1 ELSE 0 END AS is_completed,
    CASE WHEN o.invoice_amount IS NOT NULL THEN 1 ELSE 0 END AS has_invoice
    
FROM 
    automobilerepair.gold.fact_orderlifecycle o
    
    -- Join Store Information
    LEFT JOIN automobilerepair.gold.dim_store s 
        ON o.store_id = s.store_id
    
    -- Join Technician Information
    LEFT JOIN automobilerepair.gold.dim_technician t 
        ON o.technician_id = t.technician_id
    
    -- Join Survey Information
    LEFT JOIN automobilerepair.gold.dim_customer_survey cs 
        ON o.order_id = cs.order_id
    
    -- Join Estimate Information
    LEFT JOIN automobilerepair.gold.dim_estimate e 
        ON o.estimate_id = e.estimate_id
    
    -- Join Budget Information (matching on store and month)
    LEFT JOIN automobilerepair.gold.fact_budget b 
        ON o.store_id = b.store_id 
        AND DATE_FORMAT(o.invoice_date, 'yyyy-MM') = b.month

In [0]:
%sql
select * from automobilerepair.gold.cube_kpi_metric

In [0]:
row_count = spark.table("automobilerepair.gold.cube_kpi_metrics").count()
print(row_count)

In [0]:
%sql
-- KPI 1: MTD Performance vs Previous MTD
-- Month-to-date revenue and completed orders compared with previous month-to-date

WITH current_mtd AS (
    SELECT 
        store_id,
        store_name,
        manager_id,
        manager_name,
        SUM(invoice_amount) AS current_mtd_revenue,
        COUNT(CASE WHEN is_completed = 1 THEN 1 END) AS current_mtd_orders,
        CURRENT_DATE() AS reference_date,
        DAY(CURRENT_DATE()) AS days_into_month
    FROM automobilerepair.gold.cube_kpi_metrics
    WHERE invoice_date >= DATE_TRUNC('month', CURRENT_DATE())
        AND invoice_date < CURRENT_DATE()
        AND has_invoice = 1
    GROUP BY store_id, store_name, manager_id, manager_name
),
previous_mtd AS (
    SELECT 
        store_id,
        SUM(invoice_amount) AS previous_mtd_revenue,
        COUNT(CASE WHEN is_completed = 1 THEN 1 END) AS previous_mtd_orders
    FROM automobilerepair.gold.cube_kpi_metrics
    WHERE invoice_date >= DATE_TRUNC('month', ADD_MONTHS(CURRENT_DATE(), -1))
        AND invoice_date < DATE_ADD(DATE_TRUNC('month', ADD_MONTHS(CURRENT_DATE(), -1)), DAY(CURRENT_DATE()) - 1)
        AND has_invoice = 1
    GROUP BY store_id
)
SELECT 
    c.store_name,
    c.manager_name,
    c.current_mtd_revenue,
    COALESCE(p.previous_mtd_revenue, 0) AS previous_mtd_revenue,
    c.current_mtd_revenue - COALESCE(p.previous_mtd_revenue, 0) AS revenue_variance,
    CASE 
        WHEN COALESCE(p.previous_mtd_revenue, 0) > 0 THEN 
            ROUND((c.current_mtd_revenue - p.previous_mtd_revenue) * 100.0 / p.previous_mtd_revenue, 2)
        ELSE NULL 
    END AS revenue_variance_pct,
    c.current_mtd_orders,
    COALESCE(p.previous_mtd_orders, 0) AS previous_mtd_orders,
    c.current_mtd_orders - COALESCE(p.previous_mtd_orders, 0) AS orders_variance,
    CASE 
        WHEN COALESCE(p.previous_mtd_orders, 0) > 0 THEN 
            ROUND((c.current_mtd_orders - p.previous_mtd_orders) * 100.0 / p.previous_mtd_orders, 2)
        ELSE NULL 
    END AS orders_variance_pct,
    c.days_into_month
FROM current_mtd c
LEFT JOIN previous_mtd p ON c.store_id = p.store_id
ORDER BY revenue_variance_pct DESC NULLS LAST

In [0]:
%sql
-- KPI 2: Average Days in Shop
-- Average days a vehicle spends in the shop, by store and service type

SELECT 
    store_name,
    store_type,
    manager_name,
    service_type,
    COUNT(*) AS total_orders,
    ROUND(AVG(days_in_shop), 2) AS avg_days_in_shop,
    MIN(days_in_shop) AS min_days_in_shop,
    MAX(days_in_shop) AS max_days_in_shop
FROM 
    automobilerepair.gold.cube_kpi_metrics
WHERE 
    vehicle_in_datetime IS NOT NULL 
    AND vehicle_out_datetime IS NOT NULL
    AND days_in_shop IS NOT NULL
    AND days_in_shop >= 0
GROUP BY 
    store_name, store_type, manager_name, service_type
ORDER BY 
    avg_days_in_shop DESC

In [0]:
%sql
-- KPI 3: Survey Coverage
-- Number of surveys sent versus number of surveys responded, by store

SELECT 
    store_name,
    store_type,
    manager_name,
    COUNT(DISTINCT survey_id) AS surveys_sent,
    SUM(CASE WHEN survey_responded_flag = true THEN 1 ELSE 0 END) AS surveys_responded,
    COUNT(DISTINCT survey_id) - SUM(CASE WHEN survey_responded_flag = true THEN 1 ELSE 0 END) AS surveys_not_responded,
    ROUND(SUM(CASE WHEN survey_responded_flag = true THEN 1 ELSE 0 END) * 100.0 / COUNT(DISTINCT survey_id), 2) AS response_rate_pct
FROM 
    automobilerepair.gold.cube_kpi_metrics
WHERE 
    survey_id IS NOT NULL
GROUP BY 
    store_name, store_type, manager_name
ORDER BY 
    response_rate_pct DESC

In [0]:
%sql
-- KPI 4: Survey Scores Summary
-- Overall customer survey metrics by store and ranking of stores by overall satisfaction

SELECT 
    store_name,
    store_type,
    manager_name,
    COUNT(*) AS total_surveys_responded,
    ROUND(AVG(delivered_on_time_rating), 2) AS avg_delivered_on_time,
    ROUND(AVG(work_quality_rating), 2) AS avg_work_quality,
    ROUND(AVG(cleanliness_rating), 2) AS avg_cleanliness,
    ROUND(AVG(communication_rating), 2) AS avg_communication,
    ROUND(AVG(overall_satisfaction_rating), 2) AS overall_satisfaction,
    RANK() OVER (ORDER BY AVG(overall_satisfaction_rating) DESC) AS satisfaction_rank
FROM 
    automobilerepair.gold.cube_kpi_metrics
WHERE 
    survey_responded_flag = true
    AND overall_satisfaction_rating IS NOT NULL
GROUP BY 
    store_name, store_type, manager_name
ORDER BY 
    satisfaction_rank ASC

In [0]:
%sql
-- KPI 5: Revenue vs Budget
-- Monthly revenue compared against budget by manager, and ranking of managers by budget achievement

SELECT 
    manager_id,
    manager_name,
    store_name,
    invoice_year_month,
    SUM(invoice_amount) AS actual_revenue,
    MAX(budget_amount) AS budget_amount,
    SUM(invoice_amount) - MAX(budget_amount) AS revenue_variance,
    CASE 
        WHEN MAX(budget_amount) > 0 THEN 
            ROUND(SUM(invoice_amount) * 100.0 / MAX(budget_amount), 2)
        ELSE NULL 
    END AS budget_achievement_pct,
    RANK() OVER (
        PARTITION BY invoice_year_month 
        ORDER BY 
            CASE 
                WHEN MAX(budget_amount) > 0 THEN SUM(invoice_amount) * 100.0 / MAX(budget_amount)
                ELSE 0 
            END DESC
    ) AS achievement_rank
FROM 
    automobilerepair.gold.cube_kpi_metrics
WHERE 
    has_invoice = 1
    AND invoice_year_month IS NOT NULL
    AND budget_amount IS NOT NULL
GROUP BY 
    manager_id, manager_name, store_name, invoice_year_month
ORDER BY 
    invoice_year_month DESC, achievement_rank ASC

In [0]:
%sql
-- KPI 6: Top Technicians by Completion Time Accuracy
-- For each store, top 5 technicians based on accuracy of meeting promised completion dates

WITH technician_accuracy AS (
    SELECT 
        store_name,
        store_id,
        technician_id,
        technician_name,
        COUNT(*) AS total_orders,
        ROUND(AVG(completion_time_accuracy_abs_days), 2) AS avg_accuracy_days,
        SUM(CASE WHEN completion_time_variance_days = 0 THEN 1 ELSE 0 END) AS on_time_deliveries,
        SUM(CASE WHEN completion_time_variance_days < 0 THEN 1 ELSE 0 END) AS early_deliveries,
        SUM(CASE WHEN completion_time_variance_days > 0 THEN 1 ELSE 0 END) AS late_deliveries,
        ROUND(SUM(CASE WHEN completion_time_variance_days = 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS on_time_pct
    FROM 
        automobilerepair.gold.cube_kpi_metrics
    WHERE 
        promised_delivery_datetime IS NOT NULL
        AND actual_delivery_datetime IS NOT NULL
        AND completion_time_accuracy_abs_days IS NOT NULL
        AND is_completed = 1
    GROUP BY 
        store_name, store_id, technician_id, technician_name
    HAVING 
        COUNT(*) >= 5  -- Only include technicians with at least 5 orders
),
ranked_technicians AS (
    SELECT 
        *,
        ROW_NUMBER() OVER (
            PARTITION BY store_id 
            ORDER BY avg_accuracy_days ASC, on_time_pct DESC
        ) AS rank_in_store
    FROM 
        technician_accuracy
)
SELECT 
    store_name,
    technician_name,
    total_orders,
    avg_accuracy_days,
    on_time_deliveries,
    early_deliveries,
    late_deliveries,
    on_time_pct,
    rank_in_store
FROM 
    ranked_technicians
WHERE 
    rank_in_store <= 5
ORDER BY 
    store_name, rank_in_store

In [0]:
%sql
-- KPI 7: Year-to-Date Revenue Growth
-- Year-to-date revenue compared with previous year YTD, and ranking of stores by YTD growth

WITH current_ytd AS (
    SELECT 
        store_id,
        store_name,
        manager_name,
        SUM(invoice_amount) AS current_ytd_revenue,
        COUNT(*) AS current_ytd_orders,
        YEAR(CURRENT_DATE()) AS current_year
    FROM automobilerepair.gold.cube_kpi_metrics
    WHERE invoice_year = YEAR(CURRENT_DATE())
        AND invoice_date <= CURRENT_DATE()
        AND has_invoice = 1
    GROUP BY store_id, store_name, manager_name
),
previous_ytd AS (
    SELECT 
        store_id,
        SUM(invoice_amount) AS previous_ytd_revenue,
        COUNT(*) AS previous_ytd_orders,
        YEAR(CURRENT_DATE()) - 1 AS previous_year
    FROM automobilerepair.gold.cube_kpi_metrics
    WHERE invoice_year = YEAR(CURRENT_DATE()) - 1
        AND invoice_date <= DATE_ADD(CURRENT_DATE(), -365)
        AND has_invoice = 1
    GROUP BY store_id
)
SELECT 
    c.store_name,
    c.manager_name,
    c.current_ytd_revenue,
    COALESCE(p.previous_ytd_revenue, 0) AS previous_ytd_revenue,
    c.current_ytd_revenue - COALESCE(p.previous_ytd_revenue, 0) AS revenue_growth,
    CASE 
        WHEN COALESCE(p.previous_ytd_revenue, 0) > 0 THEN 
            ROUND((c.current_ytd_revenue - p.previous_ytd_revenue) * 100.0 / p.previous_ytd_revenue, 2)
        ELSE NULL 
    END AS revenue_growth_pct,
    c.current_ytd_orders,
    COALESCE(p.previous_ytd_orders, 0) AS previous_ytd_orders,
    RANK() OVER (ORDER BY 
        CASE 
            WHEN COALESCE(p.previous_ytd_revenue, 0) > 0 THEN 
                (c.current_ytd_revenue - p.previous_ytd_revenue) * 100.0 / p.previous_ytd_revenue
            ELSE 0 
        END DESC
    ) AS growth_rank
FROM current_ytd c
LEFT JOIN previous_ytd p ON c.store_id = p.store_id
ORDER BY growth_rank ASC

In [0]:
%sql
-- KPI 8: Stage-wise Day Cycle Time
-- Average days spent in each stage (vehicle-in to work-start, work-start to completion, completion to delivery)
-- By store and service type

SELECT 
    store_name,
    store_type,
    manager_name,
    service_type,
    COUNT(*) AS total_orders,
    ROUND(AVG(days_vehicle_in_to_work_start), 2) AS avg_days_vehicle_in_to_work_start,
    ROUND(AVG(days_work_start_to_completion), 2) AS avg_days_work_start_to_completion,
    ROUND(AVG(days_completion_to_delivery), 2) AS avg_days_completion_to_delivery,
    ROUND(AVG(days_in_shop), 2) AS avg_total_days_in_shop,
    ROUND(AVG(days_vehicle_in_to_work_start + days_work_start_to_completion + days_completion_to_delivery), 2) AS avg_total_cycle_time
FROM 
    automobilerepair.gold.cube_kpi_metrics
WHERE 
    days_vehicle_in_to_work_start IS NOT NULL
    AND days_work_start_to_completion IS NOT NULL
    AND days_completion_to_delivery IS NOT NULL
    AND days_vehicle_in_to_work_start >= 0
    AND days_work_start_to_completion >= 0
    AND days_completion_to_delivery >= 0
    AND is_completed = 1
GROUP BY 
    store_name, store_type, manager_name, service_type
ORDER BY 
    avg_total_cycle_time DESC

In [0]:
%sql
-- KPI 9: Estimator Accuracy
-- Accuracy of estimates (initial vs final) by estimator and ranking of estimators by highest accuracy

WITH estimator_metrics AS (
    SELECT 
        estimator_id,
        estimator_name,
        estimate_type,
        COUNT(*) AS total_estimates,
        ROUND(AVG(estimate_variance_pct), 2) AS avg_variance_pct,
        ROUND(AVG(estimate_amount), 2) AS avg_estimate_amount,
        ROUND(AVG(invoice_amount), 2) AS avg_invoice_amount,
        SUM(CASE WHEN estimate_variance_pct <= 5 THEN 1 ELSE 0 END) AS estimates_within_5pct,
        SUM(CASE WHEN estimate_variance_pct <= 10 THEN 1 ELSE 0 END) AS estimates_within_10pct,
        ROUND(SUM(CASE WHEN estimate_variance_pct <= 5 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS accuracy_5pct_rate,
        ROUND(SUM(CASE WHEN estimate_variance_pct <= 10 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS accuracy_10pct_rate
    FROM 
        automobilerepair.gold.cube_kpi_metrics
    WHERE 
        estimator_id IS NOT NULL
        AND estimate_amount IS NOT NULL
        AND invoice_amount IS NOT NULL
        AND estimate_variance_pct IS NOT NULL
        AND has_invoice = 1
    GROUP BY 
        estimator_id, estimator_name, estimate_type
    HAVING 
        COUNT(*) >= 10  -- Only include estimators with at least 10 estimates
)
SELECT 
    estimator_name,
    estimator_id,
    estimate_type,
    total_estimates,
    avg_variance_pct,
    avg_estimate_amount,
    avg_invoice_amount,
    estimates_within_5pct,
    estimates_within_10pct,
    accuracy_5pct_rate,
    accuracy_10pct_rate,
    RANK() OVER (ORDER BY avg_variance_pct ASC) AS accuracy_rank
FROM 
    estimator_metrics
ORDER BY 
    accuracy_rank ASC

In [0]:
%sql
-- KPI 10: Technician Workload
-- Number of orders handled and total days in shop per technician per month
-- Ranking of technicians by workload

WITH technician_monthly_workload AS (
    SELECT 
        technician_id,
        technician_name,
        store_name,
        store_id,
        invoice_year_month,
        COUNT(*) AS orders_handled,
        SUM(days_in_shop) AS total_days_in_shop,
        ROUND(AVG(days_in_shop), 2) AS avg_days_per_order,
        COUNT(DISTINCT service_type) AS service_types_handled,
        SUM(CASE WHEN is_completed = 1 THEN 1 ELSE 0 END) AS completed_orders,
        ROUND(SUM(CASE WHEN is_completed = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS completion_rate_pct
    FROM 
        automobilerepair.gold.cube_kpi_metrics
    WHERE 
        technician_id IS NOT NULL
        AND invoice_year_month IS NOT NULL
        AND days_in_shop IS NOT NULL
    GROUP BY 
        technician_id, technician_name, store_name, store_id, invoice_year_month
)
SELECT 
    technician_name,
    technician_id,
    store_name,
    invoice_year_month,
    orders_handled,
    total_days_in_shop,
    avg_days_per_order,
    service_types_handled,
    completed_orders,
    completion_rate_pct,
    RANK() OVER (
        PARTITION BY invoice_year_month 
        ORDER BY orders_handled DESC, total_days_in_shop DESC
    ) AS workload_rank_in_month,
    RANK() OVER (
        PARTITION BY store_id, invoice_year_month 
        ORDER BY orders_handled DESC, total_days_in_shop DESC
    ) AS workload_rank_in_store
FROM 
    technician_monthly_workload
ORDER BY 
    invoice_year_month DESC, workload_rank_in_month ASC